# AgentBench-Gov: Visualization Gallery

Interactive reproduction and exploration of the publication figures.
Figures are also generated automatically by `figures/generate_figures.py`.

**Contents:**
1. Setup and data loading
2. Radar chart (multi-dimensional comparison)
3. Governance Index bar chart
4. Dimension score heatmap
5. Difficulty degradation chart
6. Failure mode pie chart
7. Size vs. GI scatter plot
8. Score distribution boxplots
9. Sub-category performance

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

ROOT = Path('..')

# Shared style
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Load data
with open(ROOT / 'results' / 'summary_results.json') as f:
    summary = json.load(f)

MODELS = list(summary.keys())
DISPLAY_NAMES = [summary[m]['display_name'] for m in MODELS]
SHORT_NAMES = [n.replace('-Instruct','').replace('-Distill','') for n in DISPLAY_NAMES]
DIMS = ['compliance', 'transparency', 'accountability', 'safety', 'reliability']
GI = [summary[m]['governance_index'] for m in MODELS]

# Color palette (colourblind-safe)
MODEL_COLORS = ['#1E88E5', '#43A047', '#FB8C00', '#E53935', '#8E24AA', '#546E7A']
DIM_COLORS   = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

print('Data loaded. Models:', SHORT_NAMES)

## 2. Radar Chart

In [ ]:
def radar_chart(summary, models, dims, colors, short_names, title='Governance Radar'):
    N = len(dims)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    for i, model in enumerate(models):
        values = [summary[model]['dimension_scores'][d] for d in dims]
        values += values[:1]
        ax.plot(angles, values, '-o', color=colors[i], linewidth=2, markersize=4, label=short_names[i])
        ax.fill(angles, values, color=colors[i], alpha=0.08)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([d.capitalize() for d in dims], fontsize=12)
    ax.set_ylim(30, 80)
    ax.set_yticks([40, 50, 60, 70])
    ax.set_yticklabels(['40', '50', '60', '70'], fontsize=8, color='gray')
    ax.grid(alpha=0.4)
    ax.set_title(title, fontweight='bold', fontsize=14, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
    plt.tight_layout()
    return fig

fig = radar_chart(summary, MODELS, DIMS, MODEL_COLORS, SHORT_NAMES)
fig.savefig('../figures/notebook_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Governance Index Rankings

In [ ]:
# Sort by GI descending
sorted_models = sorted(zip(MODELS, GI, SHORT_NAMES, MODEL_COLORS), key=lambda x: x[1], reverse=True)
s_models, s_gi, s_names, s_colors = zip(*sorted_models)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(s_names, s_gi, color=s_colors, edgecolor='white')

for bar, gi in zip(bars, s_gi):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{gi:.2f}', va='center', fontweight='bold')

ax.axvline(50, color='red', linestyle='--', alpha=0.5, label='Minimum threshold')
ax.set_xlim(40, 75)
ax.set_xlabel('Governance Index (0–100 scale)')
ax.set_title('Model Rankings: Governance Index', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Dimension Heatmap

In [ ]:
heat = pd.DataFrame(
    {d: [summary[m]['dimension_scores'][d] for m in MODELS] for d in DIMS},
    index=SHORT_NAMES
).sort_values('safety', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    heat, annot=True, fmt='.1f',
    cmap='RdYlGn', vmin=40, vmax=75,
    linewidths=0.5, linecolor='white',
    ax=ax, cbar_kws={'label': 'Score (0–100)'}
)
ax.set_title('Dimension Score Heatmap', fontweight='bold', fontsize=13)
ax.set_xlabel('Governance Dimension')
plt.tight_layout()
plt.show()

## 5. Difficulty Degradation

In [ ]:
diff_df = pd.DataFrame(
    {level: [summary[m]['difficulty_scores'][level] for m in MODELS]
     for level in ['easy', 'medium', 'hard']},
    index=SHORT_NAMES
).sort_values('easy', ascending=False)

x = np.arange(len(diff_df))
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - 0.25, diff_df['easy'], 0.25, label='Easy', color='#4CAF50', alpha=0.9)
ax.bar(x, diff_df['medium'], 0.25, label='Medium', color='#FF9800', alpha=0.9)
ax.bar(x + 0.25, diff_df['hard'], 0.25, label='Hard', color='#F44336', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(diff_df.index, rotation=20, ha='right')
ax.set_ylabel('Score (0–100)')
ax.set_title('Performance by Difficulty Level', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Failure Mode Distribution

In [ ]:
failure_modes = {
    'Hallucinated\nCompliance': 27.1,
    'Missing\nContext': 22.3,
    'Overly\nRestrictive': 18.4,
    'Vague\nReasoning': 17.2,
    'Conflicting\nRules': 10.6,
    'Audit Trail\nOmission': 4.4,
}

fig, ax = plt.subplots(figsize=(8, 6))
wedge_colors = ['#F44336', '#FF9800', '#FFC107', '#8BC34A', '#2196F3', '#9C27B0']
wedges, texts, autotexts = ax.pie(
    list(failure_modes.values()),
    labels=list(failure_modes.keys()),
    autopct='%1.1f%%',
    colors=wedge_colors,
    startangle=90,
    pctdistance=0.75,
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')

ax.set_title('Failure Mode Distribution\n(All Models Combined)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Sub-Category Performance

In [ ]:
subcats = ['gdpr', 'hipaa', 'ai_act', 'financial']
sub_df = pd.DataFrame(
    {sc: [summary[m]['subcategory_scores'].get(sc, None) for m in MODELS]
     for sc in subcats},
    index=SHORT_NAMES
).dropna().sort_values('gdpr', ascending=False)

x = np.arange(len(sub_df))
fig, ax = plt.subplots(figsize=(12, 5))
width = 0.2
sub_colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
for i, (sc, col) in enumerate(zip(subcats, sub_colors)):
    ax.bar(x + (i - 1.5) * width, sub_df[sc], width,
           label=sc.upper().replace('_', ' '), color=col, alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(sub_df.index, rotation=20, ha='right')
ax.set_ylabel('Score (0–100)')
ax.set_title('Compliance Sub-Category Performance', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nSub-category means across all models:')
print(sub_df.mean().round(2).to_string())

## 8. Export All Figures

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, str(ROOT / 'figures' / 'generate_figures.py')],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)